# Bulk Scoring Demo: Fraud Detection API

**Use Case:** A banking official has 20 new transactions to check for fraud.

Instead of entering each one manually in the Streamlit app, we can score them all at once using the API directly!

---

## Why Bulk Scoring?

| Approach | Good For | Limitation |
|----------|----------|------------|
| **Streamlit App** | Single transactions, demos | Manual entry, slow for many records |
| **Direct API Call** | Bulk scoring, automation | Requires basic coding |

This notebook shows how easy it is to score multiple records programmatically.

## Step 1: Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import requests
import pandas as pd
import time

# Our deployed API URL
API_URL = "https://model-deployment2026.onrender.com"

print(f"API URL: {API_URL}")
print("Ready for bulk scoring!")

API URL: https://model-deployment2026.onrender.com
Ready for bulk scoring!


## Step 2: Create Sample Transaction Data

Let's simulate 20 new transactions that a banking official needs to check.

In [3]:
# 20 sample transactions - mix of normal and suspicious patterns
transactions = [
    # Normal transactions (low risk expected)
    {"category": "Grocery", "amount": 45.50, "age_at_transaction": 35, "days_until_card_expires": 400, "loc_delta": 0.01, "trans_volume_mavg": 50, "trans_volume_mstd": 20, "trans_freq": 2, "loc_delta_mavg": 0.01},
    {"category": "Gas", "amount": 55.00, "age_at_transaction": 42, "days_until_card_expires": 300, "loc_delta": 0.02, "trans_volume_mavg": 60, "trans_volume_mstd": 25, "trans_freq": 3, "loc_delta_mavg": 0.02},
    {"category": "Food", "amount": 28.75, "age_at_transaction": 28, "days_until_card_expires": 500, "loc_delta": 0.01, "trans_volume_mavg": 40, "trans_volume_mstd": 15, "trans_freq": 4, "loc_delta_mavg": 0.01},
    {"category": "Shopping", "amount": 89.99, "age_at_transaction": 55, "days_until_card_expires": 200, "loc_delta": 0.03, "trans_volume_mavg": 80, "trans_volume_mstd": 30, "trans_freq": 2, "loc_delta_mavg": 0.02},
    {"category": "Grocery", "amount": 120.00, "age_at_transaction": 45, "days_until_card_expires": 350, "loc_delta": 0.01, "trans_volume_mavg": 100, "trans_volume_mstd": 40, "trans_freq": 3, "loc_delta_mavg": 0.01},

    # Moderate transactions
    {"category": "Entertainment", "amount": 250.00, "age_at_transaction": 30, "days_until_card_expires": 450, "loc_delta": 0.15, "trans_volume_mavg": 150, "trans_volume_mstd": 60, "trans_freq": 2, "loc_delta_mavg": 0.10},
    {"category": "Shopping", "amount": 350.00, "age_at_transaction": 38, "days_until_card_expires": 180, "loc_delta": 0.20, "trans_volume_mavg": 200, "trans_volume_mstd": 80, "trans_freq": 1, "loc_delta_mavg": 0.15},
    {"category": "Travel", "amount": 450.00, "age_at_transaction": 50, "days_until_card_expires": 250, "loc_delta": 0.35, "trans_volume_mavg": 180, "trans_volume_mstd": 70, "trans_freq": 1, "loc_delta_mavg": 0.25},
    {"category": "Food", "amount": 180.00, "age_at_transaction": 33, "days_until_card_expires": 400, "loc_delta": 0.08, "trans_volume_mavg": 120, "trans_volume_mstd": 50, "trans_freq": 5, "loc_delta_mavg": 0.05},
    {"category": "Gas", "amount": 95.00, "age_at_transaction": 60, "days_until_card_expires": 100, "loc_delta": 0.05, "trans_volume_mavg": 70, "trans_volume_mstd": 30, "trans_freq": 4, "loc_delta_mavg": 0.03},

    # Suspicious patterns (high risk expected)
    {"category": "Shopping", "amount": 2500.00, "age_at_transaction": 25, "days_until_card_expires": 50, "loc_delta": 0.85, "trans_volume_mavg": 100, "trans_volume_mstd": 200, "trans_freq": 8, "loc_delta_mavg": 0.70},
    {"category": "Entertainment", "amount": 1800.00, "age_at_transaction": 22, "days_until_card_expires": 30, "loc_delta": 0.90, "trans_volume_mavg": 80, "trans_volume_mstd": 150, "trans_freq": 10, "loc_delta_mavg": 0.80},
    {"category": "Travel", "amount": 3500.00, "age_at_transaction": 28, "days_until_card_expires": 20, "loc_delta": 0.95, "trans_volume_mavg": 150, "trans_volume_mstd": 300, "trans_freq": 12, "loc_delta_mavg": 0.85},
    {"category": "Shopping", "amount": 4200.00, "age_at_transaction": 30, "days_until_card_expires": 15, "loc_delta": 0.88, "trans_volume_mavg": 200, "trans_volume_mstd": 400, "trans_freq": 15, "loc_delta_mavg": 0.75},
    {"category": "Grocery", "amount": 890.00, "age_at_transaction": 35, "days_until_card_expires": 40, "loc_delta": 0.75, "trans_volume_mavg": 90, "trans_volume_mstd": 180, "trans_freq": 9, "loc_delta_mavg": 0.60},

    # More mixed transactions
    {"category": "Food", "amount": 65.00, "age_at_transaction": 48, "days_until_card_expires": 280, "loc_delta": 0.04, "trans_volume_mavg": 55, "trans_volume_mstd": 22, "trans_freq": 3, "loc_delta_mavg": 0.03},
    {"category": "Gas", "amount": 42.00, "age_at_transaction": 52, "days_until_card_expires": 320, "loc_delta": 0.02, "trans_volume_mavg": 45, "trans_volume_mstd": 18, "trans_freq": 2, "loc_delta_mavg": 0.02},
    {"category": "Entertainment", "amount": 1200.00, "age_at_transaction": 26, "days_until_card_expires": 60, "loc_delta": 0.65, "trans_volume_mavg": 120, "trans_volume_mstd": 100, "trans_freq": 7, "loc_delta_mavg": 0.55},
    {"category": "Travel", "amount": 780.00, "age_at_transaction": 40, "days_until_card_expires": 150, "loc_delta": 0.45, "trans_volume_mavg": 160, "trans_volume_mstd": 65, "trans_freq": 2, "loc_delta_mavg": 0.30},
    {"category": "Shopping", "amount": 320.00, "age_at_transaction": 44, "days_until_card_expires": 220, "loc_delta": 0.12, "trans_volume_mavg": 140, "trans_volume_mstd": 55, "trans_freq": 3, "loc_delta_mavg": 0.08},
]

print(f"Created {len(transactions)} sample transactions")
print("\nSample transaction:")
transactions[0]

Created 20 sample transactions

Sample transaction:


{'category': 'Grocery',
 'amount': 45.5,
 'age_at_transaction': 35,
 'days_until_card_expires': 400,
 'loc_delta': 0.01,
 'trans_volume_mavg': 50,
 'trans_volume_mstd': 20,
 'trans_freq': 2,
 'loc_delta_mavg': 0.01}

## Step 3: Bulk Score All Transactions

Now let's call the API for each transaction and collect the results.

In [4]:
def score_transaction(transaction, api_url):
    """Score a single transaction via API."""
    try:
        response = requests.post(
            f"{api_url}/predict",
            json=transaction,
            timeout=60  # Allow time for cold start
        )
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"Status {response.status_code}"}
    except Exception as e:
        return {"error": str(e)}

# Score all transactions
print("Scoring transactions... (first request may take 30-60s due to cold start)")
print("="*60)

results = []
start_time = time.time()

for i, txn in enumerate(transactions):
    result = score_transaction(txn, API_URL)
    results.append(result)

    # Show progress
    if "error" in result:
        print(f"  [{i+1:2d}/20] ERROR: {result['error']}")
    else:
        verdict = result.get('verdict', 'Unknown')
        prob = result.get('ensemble_probability', 0) * 100
        print(f"  [{i+1:2d}/20] ${txn['amount']:>7.2f} {txn['category']:<12} -> {prob:5.1f}% ({verdict})")

elapsed = time.time() - start_time
print("="*60)
print(f"Completed {len(transactions)} transactions in {elapsed:.1f} seconds")

Scoring transactions... (first request may take 30-60s due to cold start)
  [ 1/20] $  45.50 Grocery      ->   2.0% (LEGITIMATE - Approved)
  [ 2/20] $  55.00 Gas          ->   1.9% (LEGITIMATE - Approved)
  [ 3/20] $  28.75 Food         ->   0.8% (LEGITIMATE - Approved)
  [ 4/20] $  89.99 Shopping     ->   0.8% (LEGITIMATE - Approved)
  [ 5/20] $ 120.00 Grocery      ->  48.0% (LOW RISK - Monitor)
  [ 6/20] $ 250.00 Entertainment ->   0.6% (LEGITIMATE - Approved)
  [ 7/20] $ 350.00 Shopping     ->   0.1% (LEGITIMATE - Approved)
  [ 8/20] $ 450.00 Travel       ->   1.0% (LEGITIMATE - Approved)
  [ 9/20] $ 180.00 Food         ->   5.5% (LEGITIMATE - Approved)
  [10/20] $  95.00 Gas          ->   1.5% (LEGITIMATE - Approved)
  [11/20] $2500.00 Shopping     ->   4.8% (LEGITIMATE - Approved)
  [12/20] $1800.00 Entertainment ->   2.8% (LEGITIMATE - Approved)
  [13/20] $3500.00 Travel       ->   3.0% (LEGITIMATE - Approved)
  [14/20] $4200.00 Shopping     ->   4.2% (LEGITIMATE - Approved)
  [

## Step 4: View Results as DataFrame

Let's organize the results in a clean table format.

In [5]:
# Create a summary DataFrame
summary_data = []

for i, (txn, result) in enumerate(zip(transactions, results)):
    if "error" not in result:
        summary_data.append({
            "#": i + 1,
            "Category": txn["category"],
            "Amount": f"${txn['amount']:.2f}",
            "XGB %": f"{result.get('xgboost_probability', 0)*100:.1f}%",
            "RF %": f"{result.get('random_forest_probability', 0)*100:.1f}%",
            "Ensemble %": f"{result.get('ensemble_probability', 0)*100:.1f}%",
            "Verdict": result.get('verdict', 'Unknown'),
            "Drift Warnings": len(result.get('drift_warnings', []))
        })
    else:
        summary_data.append({
            "#": i + 1,
            "Category": txn["category"],
            "Amount": f"${txn['amount']:.2f}",
            "XGB %": "ERROR",
            "RF %": "ERROR",
            "Ensemble %": "ERROR",
            "Verdict": result.get('error', 'Error'),
            "Drift Warnings": 0
        })

df_results = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("BULK SCORING RESULTS")
print("="*80)
df_results


BULK SCORING RESULTS


,#,Category,Amount,XGB %,RF %,Ensemble %,Verdict,Drift Warnings
0,1,Grocery,$45.50,1.6%,2.3%,2.0%,LEGITIMATE - Approved,0
1,2,Gas,$55.00,0.1%,3.8%,1.9%,LEGITIMATE - Approved,1
2,3,Food,$28.75,0.5%,1.0%,0.8%,LEGITIMATE - Approved,1
3,4,Shopping,$89.99,0.1%,1.5%,0.8%,LEGITIMATE - Approved,0
4,5,Grocery,$120.00,60.3%,35.7%,48.0%,LOW RISK - Monitor,1
5,6,Entertainment,$250.00,0.1%,1.0%,0.6%,LEGITIMATE - Approved,0
6,7,Shopping,$350.00,0.1%,0.0%,0.1%,LEGITIMATE - Approved,0
7,8,Travel,$450.00,0.1%,1.8%,1.0%,LEGITIMATE - Approved,0
8,9,Food,$180.00,5.1%,5.8%,5.5%,LEGITIMATE - Approved,1
9,10,Gas,$95.00,1.0%,2.0%,1.5%,LEGITIMATE - Approved,1


## Step 5: Summary Statistics

In [6]:
# Count verdicts
successful_results = [r for r in results if "error" not in r]

if successful_results:
    fraud_count = sum(1 for r in successful_results if r.get('ensemble_prediction', 0) == 1)
    legit_count = len(successful_results) - fraud_count
    drift_count = sum(1 for r in successful_results if len(r.get('drift_warnings', [])) > 0)

    print("="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Total Transactions Scored: {len(successful_results)}")
    print(f"")
    print(f"  FRAUD Detected:     {fraud_count:3d}  ({fraud_count/len(successful_results)*100:.1f}%)")
    print(f"  LEGIT Transactions: {legit_count:3d}  ({legit_count/len(successful_results)*100:.1f}%)")
    print(f"")
    print(f"  With Drift Warnings: {drift_count:3d}  ({drift_count/len(successful_results)*100:.1f}%)")
    print("="*50)
else:
    print("No successful results to summarize.")

SUMMARY
Total Transactions Scored: 20

  FRAUD Detected:       0  (0.0%)
  LEGIT Transactions:  20  (100.0%)

  With Drift Warnings:  13  (65.0%)


## Step 6: Export Results (Optional)

Save results to CSV for further analysis or reporting.

In [7]:
# Export to CSV
output_file = "/content/drive/MyDrive/2026_work/model-train_deployement/scoring/bulk_scoring_results.csv"
df_results.to_csv(output_file, index=False)
print(f"Results saved to: {output_file}")

# For Colab: download the file
try:
    from google.colab import files
    files.download(output_file)
    print("File downloaded!")
except:
    print("(Not in Colab - file saved locally)")

Results saved to: /content/drive/MyDrive/2026_work/model-train_deployement/scoring/bulk_scoring_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File downloaded!


---

## Key Takeaways

1. **API enables automation** - Score hundreds of transactions with a simple script
2. **No UI needed** - Perfect for batch processing, scheduled jobs, or integration with other systems
3. **Same model, different interfaces** - Streamlit for demos, API for production workloads
4. **Easy to extend** - Add to existing workflows, connect to databases, etc.

### Real-World Applications:
- **Daily batch scoring** - Process overnight transactions each morning
- **Integration** - Connect to banking systems via API
- **Automated alerts** - Flag high-risk transactions automatically
- **Audit trails** - Log all predictions for compliance